这个ipynb是在172.16.68.179上执行的grid search代码：搜索KRandGR, 数据保存明为KRandGR_results.csv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Task_independent_metrics import gen_KR_GR_input, RunSpnc, Evaluate_KR_GR
# from ResFunctions import *
import torch 
import torch.nn as nn
from spnc import spnc_anisotropy
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

CANDIDATES = [
    
    Path(r"C:\Users\tom\Desktop\Repository"),
    Path(r"C:\Users\Chen\Desktop\Repository"),
]
searchpaths = [p for p in CANDIDATES if p.exists()]

#tuple of repos
repos = ('machine_learning_library',)

# Add local modules and paths to local repos
from deterministic_mask import fixed_seed_mask, max_sequences_mask
import repo_tools
repo_tools.repos_path_finder(searchpaths, repos)
from single_node_res import single_node_reservoir
import ridge_regression as RR
from linear_layer import *
from mask import binary_mask
from utility import *
from NARMA10 import NARMA10
from datasets.load_TI46_digits import *
import datasets.load_TI46 as TI46
from sklearn.metrics import classification_report

In [ ]:
# Create a reservoir parameter class
class ReservoirParams:
    def __init__(self):
        # Reservoir parameters
        self.h = 0.4473502275692851
        self.theta_H = 90
        self.k_s_0 = 0
        self.phi = 45
        self.beta_prime = 20

        # Network parameters
        self.Nvirt = 30
        self.m0 = 0.007586422893538462
        self.bias = True
        self.Nwarmup = 0

        ## Params
        self.params = {
            'theta':0.5540233436467944, 
            'gamma':0.13738441393289658, 
            'delay_feedback':0, 
            'Nvirt':self.Nvirt,
            'length_warmup': self.Nwarmup,
            # 'train_sample': Ntrain*Nvirt,
            # 'test_sample': Ntest*Nvirt,
            'warmup_sample': self.Nwarmup*self.Nvirt,
            'voltage_noise': False,
            'seed_voltage_noise': 1234,
            'delta_V':0.1,
            'johnson_noise': False,
            'seed_johnson_noise': 1234,
            'mean_johnson_noise':0.0000,
            'std_johnson_noise':0.00001,
            'thermal_noise': False,
            'seed_thermal_noise': 1234,
            'lambda_ou':1.0,
            'sigma_ou':0.1
        }
    
    def update_params(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            if key in self.params:
                self.params[key] = value
            if not hasattr(self, key) and key not in self.params:
                raise AttributeError(f"ReservoirParams has no attribute or param key '{key}'")


Params = ReservoirParams()

Params.update_params(h=0.1)

Params.update_params(gamma=0.25)
print(Params.params)






In [ ]:
# Create a function for run the reservoir
def run_reservoir(all_inputs, reservoir_params, transforms):
    outputs = []
    for k, input_signal in enumerate(all_inputs):
        input_signal = input_signal.reshape(-1,1)
        S = RunSpnc(input_signal, 1, len(input_signal), reservoir_params.Nvirt, reservoir_params.m0, transforms, reservoir_params.params)
        outputs.append(S)
    States = np.stack(outputs, axis=0)

    return States

In [ ]:
# Create a function for changing the parameters
def run_experiment(param_dict:dict, reservoir_params: ReservoirParams, all_inputs) -> float:
	# update parameters
    reservoir_params.update_params(**param_dict)

    # Create the reservoir
    spn = spnc_anisotropy(reservoir_params.h, reservoir_params.theta_H, reservoir_params.k_s_0, reservoir_params.phi, reservoir_params.beta_prime,restart = True,Primep1= None)
    transforms = spn.gen_signal_slow_delayed_feedback

    # Run the reservoir
    States = run_reservoir(all_inputs, reservoir_params, transforms)

    # Calculate the MC
    KR, GR = Evaluate_KR_GR(States, Nreadouts=50, threshold=0.1)

    return KR, GR



In [ ]:
# Define the instances and signal
Params = ReservoirParams()
# Generate a signal
Nreadouts = 50
Nwash = 10
all_inputs = gen_KR_GR_input(Nreadouts, Nwash)


In [ ]:
import itertools
import os
import pandas as pd
def param_scan(param_ranges: dict, reservoir_params: ReservoirParams, signal, save_csv=True, filename=None):
    param_names = list(param_ranges.keys())
    param_values = list(param_ranges.values())
    param_combinations = list(itertools.product(*param_values))


    results_shape = [len(values) for values in param_values]
    results_KR = np.zeros(results_shape)
    results_GR = np.zeros(results_shape)

    datas = []

    for idx, combination in enumerate(param_combinations):
        param_dict = dict(zip(param_names, combination))
        KR, GR = run_experiment(param_dict, reservoir_params, signal)
        multi_idx = np.unravel_index(idx, results_shape)

        results_KR[multi_idx] = KR
        results_GR[multi_idx] = GR

        print(f"Run {param_dict} => KR: {KR:.4f}, GR: {GR:.4f}")

        data = param_dict.copy()
        data['KR'] = KR
        data['GR'] = GR
        datas.append(data)

    if save_csv:
        data_dir = os.path.join(os.getcwd(), 'KRandGR_data')
        os.makedirs(data_dir, exist_ok=True)

        if filename is None:
            filename = os.path.join(data_dir, 'KRandGR_results.csv')
        else:
            if not os.path.isabs(filename):
                filename = os.path.join(data_dir, filename)

        df = pd.DataFrame(datas)
        df.to_csv(filename, index=False)
        print(f"Results saved to {filename}")

    return results_KR, results_GR, param_names

        

In [ ]:
param_ranges = {
    'gamma': np.linspace(0.05, 0.2, 10),
    'theta': np.linspace(0.2, 0.4, 10),
    'h': np.linspace(0.3, 0.5, 10),
    # 'beta_prime': np.linspace(10, 30, 5),
    'Nvirt': [50],
    'm0': np.linspace(0.001, 0.005, 10),
}
results_KR, results_GR, param_names = param_scan(param_ranges, reservoir_params=Params, signal=all_inputs)
'''
这里Nvirt设置为50, Nwash设置为10, rest 为3
'''

In [ ]:
param_ranges = {
    'gamma': np.linspace(0.05, 0.2, 10),
    'theta': np.linspace(0.2, 0.4, 10),
    'h': np.linspace(0.3, 0.5, 10),
    # 'beta_prime': np.linspace(10, 30, 5),
    'Nvirt': [20, 50, 100, 200, 400],
    'm0': np.linspace(0.001, 0.005, 10),
}
results, param_names = param_scan(param_ranges, reservoir_params=Params, signal=signal)
'''
为了计算MC
'''